In [51]:
from services.clock_analysis import (
    compute_clock,
    compute_normalized_share
)

import pandas as pd

df = pd.read_csv("data/processed/cleaned.csv")

clock_data = compute_clock(df)

normalized_share = compute_normalized_share(
    clock_data
)

C:\Users\ashan\AppData\Local\Temp\ipykernel_17852\1977324761.py:8: DtypeWarning: Columns (0: offline) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/processed/cleaned.csv")


In [53]:
import pandas as pd
import plotly.graph_objects as go

df["ts"] = pd.to_datetime(df["ts"])
df["hour"] = df["ts"].dt.hour

# Minutes listened by each user at each hour
hourly = (
    df.groupby(["hour", "user"])["minutes"]
    .sum()
    .reset_index()
)

# Total listening time per user
user_totals = (
    df.groupby("user")["minutes"]
    .sum()
)

# Normalize within each user
hourly["share"] = hourly.apply(
    lambda r:
        r["minutes"] /
        user_totals[r["user"]]
        * 100,
    axis=1
)

pivot = (
    hourly
    .pivot(
        index="hour",
        columns="user",
        values="share"
    )
    .fillna(0)
)

pivot = pivot.reindex(
    range(24),
    fill_value=0
)

In [82]:
colors = {
    "Ashanti": "#4F6BFF",
    "Gabi": "#FF4FB8",
    "Maribel": "#FF8A3D"
}

fig = go.Figure()

for user in pivot.columns:

    fig.add_trace(
        go.Scatter(
            x=pivot.index,
            y=pivot[user],

            mode="markers",

            opacity=0.9,   # mais opacas

            marker=dict(
                color=colors[user],
                size=16,
            ),

            name=user
        )
    )

fig.update_layout(
    title="Distribution of Each User's Listening Across the Day",

    xaxis=dict(
        title="Hour of Day",
        dtick=1,
        range=[-0.5, 23.5],      # remove o -1
        showgrid=False,
        zeroline=False      # remove a linha vertical no 0
    ),

    yaxis=dict(
        title="% of User's Total Listening",
        showgrid=False,
        zeroline=False,      # remove a linha horizontal no 0
    ),

    template="plotly_dark",
    height=550
)

fig.show()

In [69]:
colors = {
    "Ashanti": "#4F6BFF",
    "Gabi": "#FF4FB8",
    "Maribel": "#FF8A3D"
}

fig = go.Figure()

for user in pivot.columns:

    fig.add_trace(
        go.Scatter(
            x=pivot.index,
            y=pivot[user],

            mode="markers",

            marker=dict(
                symbol="line-ew-open",
                size=18,
                color=colors[user],
                line=dict(
                    color=colors[user],
                    width=4
                )
            ),


            name=user
        )
    )

fig.update_layout(
    title="Distribution of Each User's Listening Across the Day",

    xaxis=dict(
        title="Hour of Day",
        dtick=1,
        showgrid=False
    ),

    yaxis=dict(
        title="% of User's Total Listening",
        showgrid=False
    ),

    template="plotly_dark",
    height=550
)

fig.show()